# 02 – Pre-processing Validation

Validates each step of the ECG pre-processing pipeline visually and quantitatively.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
import config
from src.preprocessor import ECGPreprocessor
from src.utils import setup_logging, plot_ecg_signal, plot_beat

setup_logging('INFO')

In [ ]:
# Generate a synthetic ECG for demonstration
def make_synthetic_ecg(fs=360, duration_s=10):
    n = duration_s * fs
    t = np.linspace(0, duration_s, n)
    ecg = np.zeros(n)
    beat_period = int(fs / 1.2)
    half = beat_period // 2
    t_beat = np.arange(-half, half + 1)
    qrs = 1.0 * np.exp(-0.5 * (t_beat / (0.02*fs))**2)
    idx = half
    while idx + half < n:
        ecg[idx-half:idx+half+1] += qrs
        idx += beat_period
    noise = np.random.RandomState(0).normal(0, 0.03, n)
    wander = 0.15 * np.sin(2 * np.pi * 0.2 * t)
    return (ecg + noise + wander).astype(np.float32)

fs = 360
raw = make_synthetic_ecg(fs=fs)
print('Signal length:', len(raw))

In [ ]:
# Step-by-step pre-processing
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
t = np.arange(len(raw)) / fs

axes[0].plot(t, raw, lw=0.8); axes[0].set_title('Raw signal')

filtered = ECGPreprocessor.bandpass_filter(raw, fs=fs)
axes[1].plot(t, filtered, lw=0.8, color='orange'); axes[1].set_title('After bandpass filter (0.5–45 Hz)')

no_wander = ECGPreprocessor.remove_baseline_wander(filtered, fs=fs)
axes[2].plot(t, no_wander, lw=0.8, color='green'); axes[2].set_title('After baseline wander removal')

normed = ECGPreprocessor.normalize_signal(no_wander)
axes[3].plot(t, normed, lw=0.8, color='red'); axes[3].set_title('After z-score normalisation')

plt.xlabel('Time (s)')
plt.tight_layout()
plt.show()
print(f'Mean={normed.mean():.4f}, Std={normed.std():.4f}')

In [ ]:
# R-peak detection
clean = ECGPreprocessor.preprocess_record(raw, fs=fs)
r_peaks = ECGPreprocessor.detect_r_peaks(clean, fs=fs)
print(f'Detected {len(r_peaks)} R-peaks')

plot_ecg_signal(clean, fs=fs, r_peaks=r_peaks, title='Pre-processed ECG with R-peaks')
plt.show()

In [ ]:
# Beat segmentation
beats, valid_idx = ECGPreprocessor.segment_beats(clean, r_peaks, fs=fs)
print(f'Segmented {len(beats)} beats of shape {beats.shape}')

fig, axes = plt.subplots(1, min(5, len(beats)), figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.plot(beats[i])
    ax.set_title(f'Beat {i}')
    ax.axis('off')
plt.suptitle('Sample beat waveforms')
plt.tight_layout()
plt.show()